# Federated AER sandbox — train, assert, latents, prototypes

Runs `fedwater`'s **actual FL protocol** (`fl_training.federated.FPLTrainer` — FedAvg + FPL
hierarchical prototypes over AER) on any world, then interrogates what it produced.

Three questions, three sections:

1. **§5 assert** — are the outputs well-formed and mutually consistent? (latent dim, row counts,
   client/global weight sync after FedAvg, prototypes == eval-mode mean latent).
2. **§6 latents** — is the latent space legible? separability of district / month, per round,
   including the untrained init (`round = -1`).
3. **§7 prototypes** — compiled into one tidy table: local (pre-FedAvg), global (FINCH clusters),
   post-FedAvg; plus drift signals, FINCH cluster counts, client similarity.

Nothing in the protocol is re-implemented — `SnapshotFPLTrainer` only adds latent snapshots via
two hooks. Run from `notebooks/prospection/`.

## 0. Setup

In [1]:
from __future__ import annotations

import copy, json, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore", category=UserWarning)

NB_DIR = Path.cwd()
SANDBOX = NB_DIR / "fed_sandbox"
SANDBOX.mkdir(exist_ok=True)


def find_repo_root(start: Path | None = None) -> Path | None:
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "conf" / "base" / "parameters.yml").exists() and (cand / "src" / "fedwater").is_dir():
            return cand
    return None


ROOT = find_repo_root()
if ROOT is None:
    raise RuntimeError(
        "fedwater repo root not found (conf/base/parameters.yml + src/fedwater). "
        "Put this notebook in <repo>/notebooks/prospection/."
    )
sys.path.insert(0, str(ROOT / "src"))

# THE code under test — imported, never re-implemented.
from fedwater.pipelines.fl_preprocessing.nodes import preprocess_clients
from fedwater.pipelines.fl_training.aer import AER
from fedwater.pipelines.fl_training.federated import (
    FPLTrainer, aggregate_prototypes, extract_prototypes)
from fedwater.pipelines.fl_training.nodes import effective_fl_seed
from fedwater.pipelines.drift_detection.nodes import compute_drift_signals
from fedwater.pipelines.personalization.nodes import _deconfounded_prototypes

import yaml
BASE_PARAMS = yaml.safe_load((ROOT / "conf" / "base" / "parameters.yml").read_text())
TIME = BASE_PARAMS["time"]          # resolution_h, days_per_month, n_months
FL_BASE = BASE_PARAMS["fl"]
SEED = BASE_PARAMS.get("seed", 42)

print(f"repo    : {ROOT}")
print(f"sandbox : {SANDBOX}")
print(f"torch   : {torch.__version__}   cuda: {torch.cuda.is_available()}")
print(f"time    : {TIME}   seed: {SEED}")

repo    : C:\Users\arthu\USPy\10_Mestrado\fedWater
sandbox : c:\Users\arthu\USPy\10_Mestrado\fedWater\notebooks\prospection\fed_sandbox
torch   : 2.5.1   cuda: True
time    : {'n_months': 24, 'days_per_month': 30, 'resolution_h': 1}   seed: 42


## 1. Worlds

Same discovery as the central sandbox: the experiments engine's cache
(`data/09_experiments/worlds/<sim_hash>/`) plus `"local"` (`data/07_model_output/clients/`).

In [2]:
def list_worlds() -> dict[str, dict]:
    """world_id -> {client_dir, label, meta}. `world_id` is `sim_hash`, or `"local"`."""
    worlds: dict[str, dict] = {}

    local_dir = ROOT / "data" / "07_model_output" / "clients"
    if local_dir.exists():
        worlds["local"] = {"client_dir": local_dir, "label": "local (data/07_model_output)",
                           "meta": {}}

    exp_root = ROOT / "data" / "09_experiments" / "worlds"
    for mpath in sorted(exp_root.glob("*/manifest.json")):
        try:
            man = json.loads(mpath.read_text())
        except (OSError, json.JSONDecodeError):
            continue
        if man.get("status") != "ok":
            continue
        sim_hash = man.get("sim_hash", mpath.parent.name)
        cdir = mpath.parent / "clone" / "data" / "07_model_output" / "clients"
        if not cdir.exists():
            continue
        meta = man.get("world", {})
        tag = meta.get("drift_district") or meta.get("consumption_map") or "?"
        worlds[sim_hash] = {"client_dir": cdir, "meta": meta,
                            "label": f"{sim_hash}  ·  {meta.get('variant', '?')}  ·  {tag}"}
    return worlds


WORLDS = list_worlds()
if not WORLDS:
    raise RuntimeError("no worlds found (no local data/07_model_output/clients, no experiments cache)")

pd.DataFrame([{"world_id": k, "label": v["label"], **v["meta"]} for k, v in WORLDS.items()]) \
    .set_index("world_id")

,label,anchor_scale,beta,close_fraction,consumption_map,drift_district,drift_seed_node,drift_to_income,drift_to_land_use,n_months,sim_seed,variant
world_id,,,,,,,,,,,,
local,local (data/07_model_output),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
00ac315559fc,00ac315559fc · baseline · District_E,0.05,0.35,0.5,LC_MR_LR_LR_LI,District_E,7,medium,industrial,30.0,42.0,baseline
07fdd4c1ecfe,07fdd4c1ecfe · baseline · District_D,0.05,0.35,0.5,LC_MR_LR_LR_LI,District_D,2,medium,residential,30.0,42.0,baseline
2c022bf89aee,2c022bf89aee · baseline · District_A,0.05,0.35,0.5,LC_MR_LR_LR_LI,District_A,61,medium,commercial,30.0,42.0,baseline
2d3d24affd4d,2d3d24affd4d · baseline · District_B,0.05,0.35,0.5,LC_MR_LR_LR_LI,District_B,93,low,commercial,30.0,42.0,baseline
3568b445db73,3568b445db73 · baseline · District_B,0.05,0.35,0.5,LC_MR_LR_LR_LI,District_B,93,high,residential,30.0,42.0,baseline
37a620a57318,37a620a57318 · baseline · District_B,0.05,0.35,0.5,LC_MR_LR_LR_LI,District_B,93,high,residential,30.0,42.0,baseline
3e948da9de32,3e948da9de32 · baseline · District_C,0.05,0.35,0.5,LC_MR_LR_LR_LI,District_C,59,low,commercial,30.0,42.0,baseline
490873baab8a,490873baab8a · baseline · District_D,0.05,0.35,0.5,LC_MR_LR_LR_LI,District_D,2,medium,residential,30.0,42.0,baseline


## 2. Configuration

`CONFIG["fl"]` mirrors `params:fl` **exactly** — it is handed to `preprocess_clients` and
`FPLTrainer` unchanged, so this bench and `kedro run` see the same object. Everything outside
`fl` is notebook-only (world choice, snapshot dials, subsampling).

In [17]:
CONFIG = {
    "world":    list(WORLDS.keys())[4],   # key into WORLDS (Section 1)
    "run_name": None,                     # None -> auto id from the hyperparameters
    "clients":  None,                     # None = all; or e.g. ["District_A", "District_D"]
    "max_windows_per_client": None,       # int (or None) -> even subsample after windowing

    "fl": {                               # === shape of params:fl ===
        "preprocessing": {
            "interval_agg_h":   2,        # hours per aggregated step (must divide the month)
            "window_size":      24,       # aggregated steps per window
            "step_size":        6,        # window stride, in aggregated steps
            "reference_months": 2,        # commissioning months the scaler is fitted on
            "label_threshold":  0.75,     # majority share needed to label a window with a month
            "feature_range":    [-1.0, 1.0],
        },
        "model": {
            "lstm_units": 30,             # latent dim = 2 * lstm_units
            "reg_ratio":  0.5,            # AER loss: (r/2)*back + (1-r)*recon + (r/2)*forward
        },
        "training": {
            "seed":                42,    # FL seed (init, shuffling, FINCH order); None -> SEED
            "rounds":              15,    # communication rounds
            "local_epochs":        1,
            "learning_rate":       1e-3,
            "batch_size":          64,
            "participation":       1.0,   # fraction of clients online per round
            "averaging":           "equal",   # equal | weight (by client window count)
            "proto_alpha":         0.2,   # FPL loss: a*InfoNCE + (1-a)*MSE to mean prototype
            "infonce_temperature": 0.02,
            "device":              "auto",    # auto | cpu | cuda
        },
    },

    "snapshot": {"every": 1, "points": 1500},   # latent snapshots; every=0 disables
}

if CONFIG["world"] not in WORLDS:
    raise KeyError(f"world {CONFIG['world']!r} not in WORLDS: {sorted(WORLDS)}")

_dev = CONFIG["fl"]["training"]["device"]
CONFIG["fl"]["training"]["device"] = \
    ("cuda" if torch.cuda.is_available() else "cpu") if _dev == "auto" else _dev
DEVICE = torch.device(CONFIG["fl"]["training"]["device"])

print("world :", WORLDS[CONFIG["world"]]["label"])
print("device:", DEVICE, "| latent dim:", 2 * CONFIG["fl"]["model"]["lstm_units"],
      "| rounds:", CONFIG["fl"]["training"]["rounds"])

world : 2d3d24affd4d  ·  baseline  ·  District_B
device: cuda | latent dim: 60 | rounds: 15


## 3. Data → windows

`preprocess_clients` verbatim. Unlike the central bench nothing is pooled — `fl_windows` stays
per-client, which is what `FPLTrainer` consumes. `META` is the calendar decoding of
`window_start_step`, joinable on `(district, window)`.

In [18]:
def load_clients(client_dir: Path, only: list[str] | None = None) -> dict[str, pd.DataFrame]:
    files = sorted(client_dir.glob("District_*.csv"))
    if not files:
        raise FileNotFoundError(
            f"No client CSVs in {client_dir}. For 'local', run `kedro run` first; for an "
            f"experiment world, build it via the experiments engine first."
        )
    dfs = {f.stem: pd.read_csv(f) for f in files}
    if only:
        missing = set(only) - set(dfs)
        if missing:
            raise KeyError(f"Unknown clients: {sorted(missing)} (have {sorted(dfs)})")
        dfs = {k: dfs[k] for k in only}
    return dfs


def build_windows(client_dfs: dict, cfg: dict):
    """fl_preprocessing's recipe + optional subsample. -> (fl_windows, scalers, report)."""
    windows, scalers, report = preprocess_clients(client_dfs, cfg["fl"], TIME)

    cap = cfg["max_windows_per_client"]
    if cap:
        for c, d in windows.items():
            n = len(d["windows"])
            if n > cap:
                idx = np.linspace(0, n - 1, cap).round().astype(int)   # even in time
                d["windows"] = d["windows"][idx]
                d["labels"] = d["labels"][idx]
                d["window_start_step"] = d["window_start_step"][idx]
    return windows, scalers, report


def window_metadata(fl_windows: dict) -> pd.DataFrame:
    """One row per window, keyed (district, window) — the calendar decoding."""
    res_h, dpm = TIME["resolution_h"], TIME["days_per_month"]
    rows = []
    for c in sorted(fl_windows):
        d = fl_windows[c]
        start_h = d["window_start_step"] * res_h              # hours since t0
        day = start_h // 24
        rows.append(pd.DataFrame({
            "district": c,
            "window": d["window_start_step"],
            "month": d["labels"],
            "hour": (start_h % 24).astype(int),
            "day": day.astype(int),
            "weekday": (day % 7).astype(int),                 # model calendar (30-day months)
            "day_of_month": (day % dpm).astype(int),
        }))
    return pd.concat(rows, ignore_index=True)


client_dfs = load_clients(WORLDS[CONFIG["world"]]["client_dir"], CONFIG["clients"])
fl_windows, fl_scalers, prep_report = build_windows(client_dfs, CONFIG)
META = window_metadata(fl_windows)

for c, d in fl_windows.items():
    print(f"{c}: windows {d['windows'].shape}  months {d['labels'].min()}..{d['labels'].max()}  "
          f"sensors {d['sensors']}")
print(f"\ntotal {len(META)} windows across {len(fl_windows)} clients "
      f"({CONFIG['fl']['preprocessing']['interval_agg_h']}h steps, "
      f"{list(fl_windows.values())[0]['windows'].shape[1] * CONFIG['fl']['preprocessing']['interval_agg_h'] / 24:.1f} d/window)")
META.groupby("district").agg(n=("window", "size"), months=("month", "nunique"))

District_A: windows (1768, 24, 5)  months 0..29  sensors ['p_64', 'p_75', 'q_100', 'q_102', 'q_85']
District_B: windows (1768, 24, 5)  months 0..29  sensors ['p_79', 'p_86', 'q_135', 'q_157', 'q_165']
District_C: windows (1768, 24, 5)  months 0..29  sensors ['p_58', 'p_74', 'q_110', 'q_119', 'q_91']
District_D: windows (1768, 24, 5)  months 0..29  sensors ['p_26', 'p_31', 'q_38', 'q_69', 'q_96']
District_E: windows (1768, 24, 5)  months 0..29  sensors ['p_12', 'p_41', 'q_11', 'q_19', 'q_59']

total 8840 windows across 5 clients (2h steps, 2.0 d/window)


,n,months
district,,
District_A,1768,30
District_B,1768,30
District_C,1768,30
District_D,1768,30
District_E,1768,30


## 4. Federated training

`SnapshotFPLTrainer` **is** `FPLTrainer` — same protocol, same seeding, same FedAvg. It only adds
two hooks: a latent snapshot after each client's local update (post-local, **pre-FedAvg** — the
personalized view), and a per-round print in `_fedavg`. `round = -1` is the shared untrained init.

`run_federated` assembles the same five artifacts the `train_federated` node emits, so anything
downstream (`drift_detection`, `dependence_detection`, `personalization`) consumes them as-is.

In [19]:
class SnapshotFPLTrainer(FPLTrainer):
    """FPLTrainer + fixed-subset latent snapshots. Protocol untouched."""

    def __init__(self, client_windows, fl, seed, every=1, points=1500, verbose=True):
        super().__init__(client_windows, fl, seed)
        self.starts = {c: client_windows[c]["window_start_step"] for c in self.clients}
        self.snap_every, self.verbose = int(every or 0), verbose
        self.n_rounds, self.snap_rows = 0, []

        rng = np.random.default_rng(seed)
        per = max(1, int(points) // len(self.clients))
        self.snap_idx = {
            c: np.sort(rng.choice(len(self.data[c][0]), min(per, len(self.data[c][0])),
                                  replace=False))
            for c in self.clients
        }
        if self.snap_every:
            for c in self.clients:
                self._snapshot(c, -1)                      # shared untrained init

    @torch.no_grad()
    def _snapshot(self, client: str, round_idx: int):
        model, bs = self.models[client], self.cfg_t["batch_size"]
        mode = model.training
        model.eval()
        w = self.data[client][0][self.snap_idx[client]]
        z = torch.cat([model.encode(w[i:i + bs, 1:-1])
                       for i in range(0, len(w), bs)]).cpu().numpy()
        model.train(mode)

        idx = self.snap_idx[client]
        df = pd.DataFrame(z, columns=[f"f{i}" for i in range(z.shape[1])])
        df.insert(0, "month", self.data[client][1].cpu().numpy()[idx])
        df.insert(0, "window", self.starts[client][idx])
        df.insert(0, "round", round_idx)
        df.insert(0, "kind", "aer_latent")
        df.insert(0, "district", client)
        self.snap_rows.append(df)

    # ------------------------------------------------------------ hooks
    def _local_update(self, client, round_idx):
        protos = super()._local_update(client, round_idx)
        if self.snap_every and (round_idx % self.snap_every == 0
                                or round_idx == self.n_rounds - 1):
            self._snapshot(client, round_idx)
        return protos

    def _fedavg(self, online):
        super()._fedavg(online)
        if self.verbose and self.log_rows:
            r = self.log_rows[-1]["round"]
            sub = [x for x in self.log_rows if x["round"] == r]
            print(f"round {r:>3}  loss {np.mean([x['loss'] for x in sub]):.5f}"
                  f"  mse {np.mean([x['loss_mse'] for x in sub]):.5f}"
                  f"  proto {np.mean([x['loss_proto'] for x in sub]):.5f}"
                  f"  | online {len(online)}/{len(self.clients)}"
                  f"  protos {sum(len(v[0]) for v in self.global_protos.values())}")

    def train(self, rounds):
        self.n_rounds = rounds
        return super().train(rounds)

    def snapshots(self):
        return pd.concat(self.snap_rows, ignore_index=True) if self.snap_rows else None


def run_federated(fl_windows: dict, cfg: dict, verbose: bool = True) -> dict:
    """Mirrors fl_training.nodes.train_federated, plus snapshots."""
    fl = cfg["fl"]
    seed = effective_fl_seed(fl, SEED)
    trainer = SnapshotFPLTrainer(fl_windows, fl, seed, verbose=verbose,
                                 **{"every": cfg["snapshot"]["every"],
                                    "points": cfg["snapshot"]["points"]})
    t0 = time.time()
    trainer.train(fl["training"]["rounds"])
    elapsed = time.time() - t0

    log = pd.DataFrame(trainer.log_rows)
    if not np.isfinite(log["loss"]).all():
        raise AssertionError("Non-finite training loss encountered.")

    return {
        "trainer": trainer,
        "seed": seed,
        "seconds": elapsed,
        "training_log": log,
        "prototype_history": pd.DataFrame(trainer.local_proto_rows),
        "global_prototype_history": pd.DataFrame(trainer.global_proto_rows),
        "latent_trajectories": trainer.latent_trajectories(fl_windows),
        "snapshots": trainer.snapshots(),
    }


RES = run_federated(fl_windows, CONFIG)
print(f"\n{RES['seconds']:.1f}s  |  seed {RES['seed']}")
{k: v.shape for k, v in RES.items() if isinstance(v, pd.DataFrame)}

round   0  loss 0.39906  mse 0.39906  proto 0.00000  | online 5/5  protos 60
round   1  loss 1.01337  mse 0.36622  proto 0.64715  | online 5/5  protos 53
round   2  loss 0.96935  mse 0.34148  proto 0.62788  | online 5/5  protos 52
round   3  loss 0.93476  mse 0.32349  proto 0.61127  | online 5/5  protos 50
round   4  loss 0.90539  mse 0.30467  proto 0.60072  | online 5/5  protos 52
round   5  loss 0.87264  mse 0.28339  proto 0.58925  | online 5/5  protos 56
round   6  loss 0.82193  mse 0.24671  proto 0.57522  | online 5/5  protos 53
round   7  loss 0.73525  mse 0.18555  proto 0.54970  | online 5/5  protos 46
round   8  loss 0.67303  mse 0.15369  proto 0.51934  | online 5/5  protos 51
round   9  loss 0.65006  mse 0.14347  proto 0.50659  | online 5/5  protos 43
round  10  loss 0.62584  mse 0.13615  proto 0.48969  | online 5/5  protos 41
round  11  loss 0.61564  mse 0.13063  proto 0.48501  | online 5/5  protos 50
round  12  loss 0.60549  mse 0.12484  proto 0.48065  | online 5/5  protos 39

{'training_log': (75, 6),
 'prototype_history': (2250, 63),
 'global_prototype_history': (731, 63),
 'latent_trajectories': (8840, 64),
 'snapshots': (24000, 65)}

## 5. Assert the outputs

Structural + semantic checks. The one that matters: `extract_prototypes` on a client's final model
must reproduce the per-month mean of `latent_trajectories` — i.e. prototypes and the latent table
are the same object, computed the same way, in eval mode.

Note the asymmetry the protocol creates: `prototype_history`'s final round is **pre-FedAvg**
(per-client weights), while `latent_trajectories` is **post-FedAvg** (every client holds the
global weights). §7 quantifies the gap.

In [20]:
def check_outputs(res: dict, fl_windows: dict, cfg: dict) -> pd.DataFrame:
    tr, fl = res["trainer"], cfg["fl"]
    D = 2 * fl["model"]["lstm_units"]
    lt, ph, gph = res["latent_trajectories"], res["prototype_history"], res["global_prototype_history"]
    fcols = [c for c in lt.columns if c.startswith("f")]
    pcols = [c for c in ph.columns if c.startswith("f")]

    n_win = sum(len(d["windows"]) for d in fl_windows.values())
    n_months = {c: len(np.unique(d["labels"])) for c, d in fl_windows.items()}
    rounds = fl["training"]["rounds"]
    exp_proto = rounds * sum(n_months.values()) if fl["training"]["participation"] == 1.0 else None

    gw = tr.global_model.state_dict()
    synced = all(torch.equal(gw[k], tr.models[c].state_dict()[k])
                 for c in tr.clients for k in gw)

    # prototypes == eval-mode per-month mean latent, on the SAME weights
    c0 = tr.clients[0]
    w0, l0 = tr.data[c0]
    p0 = extract_prototypes(tr.models[c0], w0, l0.cpu().numpy(), fl["training"]["batch_size"])
    g0 = lt[lt["district"] == c0].groupby("month")[fcols].mean()
    proto_err = max(float(np.abs(p0[m] - g0.loc[m].to_numpy()).max()) for m in p0)

    lat_ok = bool(np.isfinite(lt[fcols].to_numpy()).all())
    pro_ok = bool(np.isfinite(ph[pcols].to_numpy()).all())

    checks = [
        ("latent_dim",              len(fcols),                 len(fcols) == D),
        ("prototype_dim",           len(pcols),                 len(pcols) == D),
        ("latent_rows == windows",  len(lt),                    len(lt) == n_win),
        ("latents_finite",          lat_ok,                     lat_ok),
        ("prototypes_finite",       pro_ok,                     pro_ok),
        ("proto_rows",              len(ph),                    exp_proto is None or len(ph) == exp_proto),
        ("proto_rounds",            ph["round"].nunique(),      ph["round"].nunique() == rounds),
        ("global_proto_rounds",     gph["round"].nunique(),     gph["round"].nunique() == rounds),
        ("clients_synced_to_global", synced,                    synced),
        ("proto == mean latent",    f"{proto_err:.2e}",         proto_err < 1e-4),
        ("log_rows",                len(res["training_log"]),
         len(res["training_log"]) == rounds * len(tr.clients) * fl["training"]["local_epochs"]),
    ]
    out = pd.DataFrame(checks, columns=["check", "value", "passed"])
    if not out["passed"].all():
        print(out[~out["passed"]].to_string(index=False))
    return out


CHECKS = check_outputs(RES, fl_windows, CONFIG)
CHECKS

,check,value,passed
0,latent_dim,60,True
1,prototype_dim,60,True
2,latent_rows == windows,8840,True
3,latents_finite,True,True
4,prototypes_finite,True,True
5,proto_rows,2250,True
6,proto_rounds,15,True
7,global_proto_rounds,15,True
8,clients_synced_to_global,True,True
9,proto == mean latent,2.98e-07,True


In [21]:
# loss trajectory — per round, averaged over clients
RES["training_log"].groupby("round")[["loss", "loss_mse", "loss_proto"]].mean()

,loss,loss_mse,loss_proto
round,,,
0,0.399061,0.399061,0.000000
1,1.013375,0.366221,0.647154
2,0.969354,0.341476,0.627877
3,0.934761,0.323489,0.611272
4,0.905386,0.304669,0.600716
5,0.872636,0.283388,0.589248
6,0.821934,0.246709,0.575225
7,0.735255,0.185550,0.549704
8,0.673027,0.153689,0.519338


## 6. Latent space

`SNAPS` is the fixed-subset latent at every round (`-1` = untrained init), so the same windows can
be followed. Metrics per round:

| metric | reads |
|---|---|
| `sil_district` / `sil_month` | cosine silhouette — is the space *clustered* by client / by month |
| `probe_*` vs `chance_*` | linear probe accuracy — is the label *linearly decodable* |
| `pca90` / `eff_dim` | components for 90 % variance; participation ratio of the spectrum |

The round `-1` row is the control: whatever it already scores is what the architecture reads
before any training.

In [22]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def _probe(Z: np.ndarray, y: np.ndarray, seed: int = 0):
    """(held-out accuracy, majority-class share) for a linear probe."""
    _, counts = np.unique(y, return_counts=True)
    if len(counts) < 2:
        return np.nan, np.nan
    strat = y if counts.min() >= 2 else None
    Ztr, Zte, ytr, yte = train_test_split(Z, y, test_size=0.3, random_state=seed, stratify=strat)
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    clf.fit(Ztr, ytr)
    chance = float(pd.Series(yte).value_counts(normalize=True).max())
    return float(clf.score(Zte, yte)), chance


def latent_metrics(df: pd.DataFrame, seed: int = 0, sil_n: int = 2000,
                   probe_n: int = 12000) -> dict:
    fcols = [c for c in df.columns if c.startswith("f")]
    Z = df[fcols].to_numpy(np.float64)
    d, m = df["district"].to_numpy(), df["month"].to_numpy()

    rng = np.random.default_rng(seed)
    idx = rng.choice(len(Z), min(sil_n, len(Z)), replace=False)
    pidx = rng.choice(len(Z), min(probe_n, len(Z)), replace=False)
    ev = PCA().fit(Z).explained_variance_ratio_
    acc_d, ch_d = _probe(Z[pidx], d[pidx], seed)
    acc_m, ch_m = _probe(Z[pidx], m[pidx], seed)
    return dict(
        n=len(Z),
        sil_district=float(silhouette_score(Z[idx], d[idx], metric="cosine")),
        sil_month=float(silhouette_score(Z[idx], m[idx], metric="cosine")),
        probe_district=acc_d, chance_district=ch_d,
        probe_month=acc_m, chance_month=ch_m,
        pca90=int(np.searchsorted(np.cumsum(ev), 0.90) + 1),
        eff_dim=float(1.0 / (ev ** 2).sum()),
    )


SNAPS = RES["snapshots"]
BY_ROUND = None if SNAPS is None else pd.DataFrame(
    [{"round": r, **latent_metrics(g)} for r, g in SNAPS.groupby("round")]).set_index("round")
BY_ROUND if BY_ROUND is None else BY_ROUND.round(4)

,n,sil_district,sil_month,probe_district,chance_district,probe_month,chance_month,pca90,eff_dim
round,,,,,,,,,
-1,1500,0.0401,-0.2360,0.9911,0.2,0.1222,0.04,6,4.0262
0,1500,0.1123,-0.2194,1.0000,0.2,0.1289,0.04,7,4.1293
1,1500,0.0861,-0.2284,1.0000,0.2,0.1267,0.04,7,4.8740
2,1500,0.0370,-0.2306,1.0000,0.2,0.1533,0.04,7,4.2626
3,1500,0.0368,-0.2254,1.0000,0.2,0.2067,0.04,7,4.5953
4,1500,-0.0248,-0.2240,1.0000,0.2,0.2178,0.04,7,4.6073
5,1500,-0.0148,-0.2107,1.0000,0.2,0.2333,0.04,7,4.6519
6,1500,0.0141,-0.1835,1.0000,0.2,0.2667,0.04,7,5.0823
7,1500,0.0020,-0.1415,1.0000,0.2,0.3089,0.04,7,5.2758


In [23]:
# final model, ALL windows (post-FedAvg — every client holds the global weights)
cols = {"final": pd.Series(latent_metrics(RES["latent_trajectories"]))}
if BY_ROUND is not None and -1 in BY_ROUND.index:
    cols["last - untrained"] = BY_ROUND.loc[BY_ROUND.index.max()] - BY_ROUND.loc[-1]
pd.DataFrame(cols).round(4)

,final,last - untrained
n,8840.0000,0.0000
sil_district,-0.0519,-0.0619
sil_month,-0.1324,0.1381
probe_district,0.9970,0.0089
chance_district,0.2002,0.0000
probe_month,0.3869,0.2356
chance_month,0.0336,0.0000
pca90,6.0000,0.0000
eff_dim,3.7849,-0.2433


In [24]:
def dispersion(df: pd.DataFrame, by: str = "district") -> pd.DataFrame:
    """Between- vs within-group scatter of the latent — a scale-free separability ratio."""
    fcols = [c for c in df.columns if c.startswith("f")]
    Z = df[fcols].to_numpy(np.float64)
    g = df[by].to_numpy()
    grand = Z.mean(0)
    rows = []
    for u in np.unique(g):
        Zi = Z[g == u]
        rows.append(dict(**{by: u}, n=len(Zi),
                         between=float(np.linalg.norm(Zi.mean(0) - grand)),
                         within=float(np.linalg.norm(Zi - Zi.mean(0), axis=1).mean())))
    out = pd.DataFrame(rows).set_index(by)
    out["ratio"] = out["between"] / out["within"]
    return out


dispersion(RES["latent_trajectories"], "district").round(4)

,n,between,within,ratio
district,,,,
District_A,1768,0.4025,1.2479,0.3225
District_B,1768,0.2971,1.1870,0.2503
District_C,1768,0.2897,1.0423,0.2779
District_D,1768,0.3493,1.0107,0.3456
District_E,1768,0.4154,1.1406,0.3641


## 7. Prototypes

Three scopes of the same object, compiled into one tidy table:

| `scope` | what | where it comes from |
|---|---|---|
| `local` | per-client per-month mean latent, **pre-FedAvg** | `prototype_history` |
| `global` | FINCH cluster centroids per month, server side | `global_prototype_history` |
| `post_fedavg` | per-client per-month mean latent under the **global** weights | `latent_trajectories` |

`local` is what `drift_detection` / `personalization` actually consume.

In [25]:
def compile_prototypes(res: dict) -> pd.DataFrame:
    """local | global | post_fedavg prototypes in one frame."""
    ph, gph, lt = (res["prototype_history"], res["global_prototype_history"],
                   res["latent_trajectories"])
    fcols = [c for c in ph.columns if c.startswith("f")]
    last = int(ph["round"].max())

    local = ph.assign(scope="local")[["scope", "round", "client", "month"] + fcols] \
              .assign(cluster=np.nan)
    glob = gph.assign(scope="global", client=pd.NA)[
        ["scope", "round", "client", "month", "cluster"] + fcols]

    post = (lt.groupby(["district", "month"])[[c for c in lt.columns if c.startswith("f")]]
              .mean().reset_index()
              .rename(columns={"district": "client"})
              .assign(scope="post_fedavg", round=last, cluster=np.nan))
    post = post[["scope", "round", "client", "month", "cluster"] + fcols]

    return pd.concat([local[glob.columns], glob, post], ignore_index=True)


PROTOTYPES = compile_prototypes(RES)
PROTOTYPES.groupby("scope").agg(rows=("month", "size"), rounds=("round", "nunique"),
                                months=("month", "nunique"))

,rows,rounds,months
scope,,,
global,731,15,30
local,2250,15,30
post_fedavg,150,1,30


In [27]:
def _cos_matrix(P: dict[str, np.ndarray]) -> pd.DataFrame:
    """Row-wise cosine, averaged over rows, for equal-shaped per-client stacks."""
    ks = sorted(P)
    M = np.zeros((len(ks), len(ks)))
    for i, a in enumerate(ks):
        for j, b in enumerate(ks):
            A, B = np.atleast_2d(P[a]), np.atleast_2d(P[b])
            den = np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1)
            M[i, j] = float(np.mean((A * B).sum(1) / np.where(den > 0, den, np.nan)))
    return pd.DataFrame(M, index=ks, columns=ks)


def client_prototype_similarity(ph: pd.DataFrame) -> pd.DataFrame:
    """Mean over months of the per-month client-pair cosine, final round."""
    fcols = [c for c in ph.columns if c.startswith("f")]
    final = ph[ph["round"] == ph["round"].max()]
    months = sorted(set.intersection(*(set(g["month"]) for _, g in final.groupby("client"))))
    P = {c: g[g["month"].isin(months)].sort_values("month")[fcols].to_numpy()
         for c, g in final.groupby("client")}
    return _cos_matrix(P)


PH = RES["prototype_history"]
print("raw prototype similarity (final round, mean over months):")
display(client_prototype_similarity(PH).round(4))

print("\ncommon-mode removed (personalization._deconfounded_prototypes):")
display(_cos_matrix(_deconfounded_prototypes(PH)).round(4))

raw prototype similarity (final round, mean over months):


,District_A,District_B,District_C,District_D,District_E
District_A,1.0000,0.9679,0.9784,0.9746,0.9796
District_B,0.9679,1.0000,0.9779,0.9685,0.9659
District_C,0.9784,0.9779,1.0000,0.9770,0.9708
District_D,0.9746,0.9685,0.9770,1.0000,0.9771
District_E,0.9796,0.9659,0.9708,0.9771,1.0000



common-mode removed (personalization._deconfounded_prototypes):


,District_A,District_B,District_C,District_D,District_E
District_A,1.0000,-0.2367,-0.2891,-0.1960,0.2826
District_B,-0.2367,1.0000,0.2347,-0.2621,-0.3089
District_C,-0.2891,0.2347,1.0000,-0.1482,-0.4623
District_D,-0.1960,-0.2621,-0.1482,1.0000,0.1112
District_E,0.2826,-0.3089,-0.4623,0.1112,1.0000


In [ ]:
LC_MR_LR_LR_LI	District_B	93	low	commercial	

In [13]:
# FINCH cluster count per (round, month): >1 means the clients' domains for that month
# did not collapse into a single global prototype — the heterogeneity signal.
CLUSTERS = (RES["global_prototype_history"].groupby(["round", "month"])["cluster"]
            .nunique().unstack("month"))
print(f"clusters per month — mean {CLUSTERS.to_numpy().mean():.2f}, "
      f"max {CLUSTERS.to_numpy().max()}, months always-1: "
      f"{int((CLUSTERS == 1).all(axis=0).sum())}/{CLUSTERS.shape[1]}")
CLUSTERS.tail(5)

clusters per month — mean 1.96, max 2, months always-1: 0/30


month,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
round,,,,,,,,,,,,,,,,,,,,,
10,2,2,2,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2
11,2,2,2,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2
12,2,2,2,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2
13,2,2,2,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2
14,2,2,2,2,2,2,2,2,2,2,...,2,2,2,2,2,2,1,2,2,2


In [14]:
# drift signals straight off the prototypes — fl_training -> drift_detection, verbatim
DRIFT = compute_drift_signals(PH, CONFIG["fl"])
DRIFT.pivot(index="month", columns="client", values="delta_first").round(4)

client,District_A,District_B,District_C,District_D,District_E
month,,,,,
0,0.0000,0.0000,0.0000,0.0000,0.0000
1,0.0008,0.0002,0.0007,0.0024,0.0120
2,0.0212,0.0143,0.0153,0.0319,0.0285
3,0.1401,0.0690,0.0505,0.1451,0.1453
4,0.3707,0.1313,0.1178,0.3774,0.3486
5,0.5774,0.1979,0.1870,0.5153,0.4704
6,0.6076,0.2090,0.1982,0.5399,0.5480
7,0.5539,0.1904,0.1757,0.4922,0.4376
8,0.3924,0.1440,0.1240,0.3739,0.4141


In [15]:
def prototype_convergence(ph: pd.DataFrame) -> pd.DataFrame:
    """Cosine of each round's prototypes to the final round's, averaged."""
    fcols = [c for c in ph.columns if c.startswith("f")]
    ref = ph[ph["round"] == ph["round"].max()].set_index(["client", "month"])[fcols]
    rows = []
    for r, g in ph.groupby("round"):
        gg = g.set_index(["client", "month"])[fcols]
        common = ref.index.intersection(gg.index)
        A, B = gg.loc[common].to_numpy(), ref.loc[common].to_numpy()
        den = np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1)
        rows.append(dict(round=r, n=len(common),
                         cos_to_final=float(np.mean((A * B).sum(1) / den))))
    return pd.DataFrame(rows).set_index("round")


def fedavg_gap(res: dict) -> float:
    """Mean cosine distance between the pre-FedAvg (local) and post-FedAvg prototypes."""
    p = PROTOTYPES[PROTOTYPES["scope"].isin(["local", "post_fedavg"])]
    p = p[p["round"] == p["round"].max()]
    fcols = [c for c in p.columns if c.startswith("f")]
    piv = p.set_index(["scope", "client", "month"])[fcols]
    a = piv.loc["local"].sort_index().to_numpy()
    b = piv.loc["post_fedavg"].sort_index().to_numpy()
    den = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1)
    return float(np.mean(1 - (a * b).sum(1) / den))


print(f"mean cosine distance local (pre-FedAvg) -> post_fedavg prototypes: {fedavg_gap(RES):.4f}")
prototype_convergence(PH).round(4)

mean cosine distance local (pre-FedAvg) -> post_fedavg prototypes: 0.0485


,n,cos_to_final
round,,
0,150,0.8395
1,150,0.8704
2,150,0.8938
3,150,0.9164
4,150,0.9233
5,150,0.9330
6,150,0.9241
7,150,0.9304
8,150,0.9203


## 8. Save

`fed_sandbox/<world_id>/<run_id>/`: global + per-client weights, the five node artifacts, the
snapshots, the compiled prototypes, and the derived tables. One row per run in the shared
`runs.csv`.

In [ ]:
def _write_table(df: pd.DataFrame, path_base: Path) -> None:
    try:
        df.to_parquet(path_base.with_suffix(".parquet"), index=False)
    except Exception:
        df.to_csv(path_base.with_suffix(".csv.gz"), index=False)


def run_id_for(cfg: dict) -> str:
    if cfg["run_name"]:
        return cfg["run_name"]
    p, m, t = cfg["fl"]["preprocessing"], cfg["fl"]["model"], cfg["fl"]["training"]
    return (f"{time.strftime('%Y%m%d-%H%M%S')}_fed_u{m['lstm_units']}_r{t['rounds']}"
            f"_le{t['local_epochs']}_a{t['proto_alpha']}"
            f"_agg{p['interval_agg_h']}h_w{p['window_size']}_s{p['step_size']}")


def save_run(res, cfg, fl_windows, scalers, world_id, checks=None, prototypes=None,
             drift=None, by_round=None, run_id=None) -> Path:
    run_id = run_id or run_id_for(cfg)
    out = SANDBOX / world_id / run_id
    out.mkdir(parents=True, exist_ok=True)
    tr, fl = res["trainer"], cfg["fl"]

    torch.save({
        "global": tr.global_model.state_dict(),
        **{c: tr.models[c].state_dict() for c in tr.clients},
        "meta": {"lstm_units": fl["model"]["lstm_units"],
                 "window_size": tr.global_model.window_size,
                 "n_features": tr.global_model.head.out_features,
                 "latent_dim": tr.global_model.latent_dim,
                 "sensors": fl_windows[next(iter(fl_windows))]["sensors"],
                 "clients": tr.clients,
                 "seed": res["seed"]},
        "config": cfg,
        "world_id": world_id,
    }, out / "fed_model.pt")

    (out / "config.json").write_text(json.dumps(cfg, indent=2))
    res["training_log"].to_csv(out / "training_log.csv", index=False)
    scalers.to_csv(out / "scalers.csv", index=False)
    for name in ("prototype_history", "global_prototype_history", "latent_trajectories"):
        _write_table(res[name], out / name)
    if res["snapshots"] is not None:
        _write_table(res["snapshots"], out / "latents_by_round")
    for name, df in (("prototypes", prototypes), ("drift_signals", drift),
                     ("checks", checks), ("latent_by_round_metrics",
                                          None if by_round is None else by_round.reset_index())):
        if df is not None:
            _write_table(df, out / name)

    log = res["training_log"]
    last = log[log["round"] == log["round"].max()]
    row = dict(world_id=world_id, run_id=run_id,
               saved_at=pd.Timestamp.now().isoformat(timespec="seconds"),
               **{f"w_{k}": v for k, v in WORLDS.get(world_id, {}).get("meta", {}).items()},
               **{f"pp_{k}": v for k, v in fl["preprocessing"].items()},
               **{f"m_{k}": v for k, v in fl["model"].items()},
               **{f"t_{k}": v for k, v in fl["training"].items()},
               n_clients=len(tr.clients), n_windows=len(res["latent_trajectories"]),
               n_features=tr.global_model.head.out_features,
               latent_dim=tr.global_model.latent_dim,
               fl_seed=res["seed"], seconds=res["seconds"],
               final_loss=float(last["loss"].mean()),
               final_loss_mse=float(last["loss_mse"].mean()),
               final_loss_proto=float(last["loss_proto"].mean()),
               checks_passed=None if checks is None else bool(checks["passed"].all()),
               sil_district=None if by_round is None else float(by_round["sil_district"].iloc[-1]),
               sil_month=None if by_round is None else float(by_round["sil_month"].iloc[-1]),
               probe_district=None if by_round is None else float(by_round["probe_district"].iloc[-1]),
               probe_month=None if by_round is None else float(by_round["probe_month"].iloc[-1]))
    reg_path = SANDBOX / "runs.csv"
    reg = pd.concat([pd.read_csv(reg_path), pd.DataFrame([row])], ignore_index=True) \
        if reg_path.exists() else pd.DataFrame([row])
    reg.to_csv(reg_path, index=False)

    print(f"saved -> {out}")
    return out


RUN_DIR = save_run(RES, CONFIG, fl_windows, fl_scalers, CONFIG["world"],
                   checks=CHECKS, prototypes=PROTOTYPES, drift=DRIFT, by_round=BY_ROUND)
WORLD_ID, RUN_ID = CONFIG["world"], RUN_DIR.name

## 9. Load saved runs

In [ ]:
def list_runs(world_id: str | None = None) -> list[tuple[str, str]]:
    out = []
    for wdir in sorted(p for p in SANDBOX.iterdir() if p.is_dir()):
        if world_id and wdir.name != world_id:
            continue
        for rdir in sorted(p for p in wdir.iterdir() if p.is_dir()):
            if (rdir / "fed_model.pt").exists():
                out.append((wdir.name, rdir.name))
    return out


def _read_table(path_base: Path) -> pd.DataFrame | None:
    for suf in (".parquet", ".csv.gz"):
        p = path_base.with_suffix(suf)
        if p.exists():
            return pd.read_parquet(p) if suf == ".parquet" else pd.read_csv(p)
    return None


def _torch_load(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:                      # torch < 2.4
        return torch.load(path, map_location=map_location)


def load_run(world_id: str, run_id: str) -> dict:
    """-> {state, training_log, prototype_history, ..., latents_by_round, prototypes}."""
    d = SANDBOX / world_id / run_id
    out = {"state": _torch_load(d / "fed_model.pt", "cpu"),
           "training_log": pd.read_csv(d / "training_log.csv")}
    for name in ("prototype_history", "global_prototype_history", "latent_trajectories",
                 "latents_by_round", "prototypes", "drift_signals", "checks"):
        out[name] = _read_table(d / name)
    return out


def load_model(world_id: str, run_id: str, client: str = "global") -> AER:
    state = _torch_load(SANDBOX / world_id / run_id / "fed_model.pt", DEVICE)
    m = state["meta"]
    mdl = AER(m["n_features"], m["window_size"], m["lstm_units"]).to(DEVICE)
    mdl.load_state_dict(state[client])
    mdl.eval()
    return mdl


pd.read_csv(SANDBOX / "runs.csv").tail(10) if (SANDBOX / "runs.csv").exists() else None

## 10. Sweep (optional)

Trains every `(world, override)` combination; overrides are dotted into `CONFIG["fl"]`
(e.g. `"training.rounds"`, `"model.lstm_units"`, `"preprocessing.window_size"`). Each run is
saved and registered, so §9 picks them all up.

In [ ]:
def sweep(worlds: list[str], overrides: list[dict], base: dict | None = None,
          metrics: bool = True) -> list[tuple[str, str]]:
    base = base or CONFIG
    done = []
    for world_id in worlds:
        cdfs = load_clients(WORLDS[world_id]["client_dir"], base["clients"])
        for ov in overrides:
            cfg = copy.deepcopy(base)
            cfg["world"], cfg["run_name"] = world_id, None
            for dotted, val in ov.items():
                section, key = dotted.split(".")
                cfg["fl"][section][key] = val
            print(f"\n=== {world_id[:12]} · {ov} ===")
            fw, sc, _ = build_windows(cdfs, cfg)
            res = run_federated(fw, cfg, verbose=False)
            chk = check_outputs(res, fw, cfg)
            br = (pd.DataFrame([{"round": r, **latent_metrics(g)}
                                for r, g in res["snapshots"].groupby("round")]).set_index("round")
                  if metrics and res["snapshots"] is not None else None)
            rid = save_run(res, cfg, fw, sc, world_id, checks=chk,
                           prototypes=compile_prototypes(res),
                           drift=compute_drift_signals(res["prototype_history"], cfg["fl"]),
                           by_round=br).name
            print(f"    checks {'ok' if chk['passed'].all() else 'FAILED'}  "
                  f"| {res['seconds']:.1f}s")
            done.append((world_id, rid))
    return done


# sweep(
#     list(WORLDS.keys()),
#     [
#         {"training.rounds": 15, "training.proto_alpha": 0.2},
#         {"training.rounds": 15, "training.proto_alpha": 0.0},   # FedAvg only, no FPL term
#         {"training.rounds": 15, "model.lstm_units": 60},
#     ],
# )